# Run frozen factor-recoverability probes
Attach private datasets `thestonedape/task-aware-eegtotext` and `thestonedape/task-aware-eeg2text-frozen-glim-vectors`. Enable Internet and private secret `GITHUB_TOKEN`. A CPU session is sufficient. This notebook rebuilds the frozen metadata protocol, validates the immutable vector artifact, runs a real-data smoke, then evaluates length, native SR sentiment, ZuCo1 NR relation content, and TSR instruction under correct, metadata-only, task-only, matched-wrong, zero, and Gaussian controls. It never reads the test split and does not train the task-aware EEG-to-text model.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
COMMIT = 'f21c89a0cc1229a4f655361ed46843b3f6f443e7'
WORKTREE = '/kaggle/working/SemKey'
PROTOCOL = '/kaggle/working/frozen-recoverability-protocol'
SMOKE_OUTPUT = '/kaggle/working/frozen-factor-probe-smoke'
OUTPUT = '/kaggle/working/task-aware-eeg2text-frozen-factor-probes'
EXPECTED_INDEX_SHA256 = 'bdaaaf5c91d3c9eec16a0727825da996fd2186867245951bfdfdc92aab7738b0'
EXPECTED_VECTOR_INDEX_SHA256 = '65d4a1f38f0df801f17ccb5504b4ee5d2f43aedcbc051e1ae1f405684ccad0e2'
EXPECTED_VECTOR_MANIFEST_SHA256 = '4861d9439a8a1253d87f38524a3720ea61d0934934589434f869b73263d36eba'
EXPECTED_REGISTRY_SHA256 = 'e4afa8a1bc859b86ad786016400035bec6d4a5280350abe271d0d74186af6517'
EXPECTED_RECOVERABILITY_ROWS_SHA256 = '4cdb7899088d32c37671bbd2088f650213ba001a771738a66a85660feee67f7d'
assert all(len(value) == 64 for value in [EXPECTED_INDEX_SHA256, EXPECTED_VECTOR_INDEX_SHA256, EXPECTED_VECTOR_MANIFEST_SHA256, EXPECTED_REGISTRY_SHA256, EXPECTED_RECOVERABILITY_ROWS_SHA256])

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', 'numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2'], check=True)
import numpy, scipy, sklearn
from kaggle_secrets import UserSecretsClient
print({'python': platform.python_version(), 'numpy': numpy.__version__, 'scipy': scipy.__version__, 'sklearn': sklearn.__version__})
github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == COMMIT
for regression in ['test_protocol_manifests.py', 'test_recoverability_protocol.py', 'test_frozen_vector_contract.py', 'test_frozen_factor_probes.py']:
    subprocess.run([sys.executable, os.path.join(WORKTREE, 'evaluation', regression)], check=True)

In [ ]:
def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()
shard_manifests = glob.glob('/kaggle/input/**/metadata/shard_manifest.json', recursive=True)
assert len(shard_manifests) == 1, ('Attach exactly one canonical sharded dataset', shard_manifests)
dataset_root = os.path.dirname(os.path.dirname(shard_manifests[0]))
vector_manifests = glob.glob('/kaggle/input/**/vector_manifest.json', recursive=True)
assert len(vector_manifests) == 1, ('Attach exactly one frozen GLIM vector dataset', vector_manifests)
vector_root = os.path.dirname(vector_manifests[0])
assert digest(vector_manifests[0]) == EXPECTED_VECTOR_MANIFEST_SHA256
assert digest(os.path.join(vector_root, 'vector_index.csv')) == EXPECTED_VECTOR_INDEX_SHA256
vector_manifest = json.load(open(vector_manifests[0], encoding='utf-8'))
assert vector_manifest['status'] == 'pass' and vector_manifest['checks']['held_out_test_accessed'] is False
print({'dataset_root': dataset_root, 'vector_root': vector_root, 'vector_rows': sum(vector_manifest['condition_counts'].values()), 'chunks': len(vector_manifest['chunks'])})

In [ ]:
for path in (PROTOCOL, SMOKE_OUTPUT, OUTPUT):
    if os.path.exists(path):
        shutil.rmtree(path)
subprocess.run([
    sys.executable, os.path.join(WORKTREE, 'evaluation', 'build_recoverability_protocol.py'),
    '--dataset-root', dataset_root, '--output-root', PROTOCOL,
    '--expected-index-sha256', EXPECTED_INDEX_SHA256,
], check=True)
assert digest(os.path.join(PROTOCOL, 'recoverability_registry.json')) == EXPECTED_REGISTRY_SHA256
assert digest(os.path.join(PROTOCOL, 'recoverability_rows.csv')) == EXPECTED_RECOVERABILITY_ROWS_SHA256
protocol_report = json.load(open(os.path.join(PROTOCOL, 'recoverability_contract_report.json'), encoding='utf-8'))
assert protocol_report['status'] == 'pass' and protocol_report['checks']['held_out_test_accessed'] is False
print({'protocol': 'PASS', 'recoverability_rows': protocol_report['counts']['recoverability_rows'], 'subject_folds': protocol_report['counts']['subject_folds']})

In [ ]:
base_command = [
    sys.executable, os.path.join(WORKTREE, 'evaluation', 'run_frozen_factor_probes.py'),
    '--vector-root', vector_root, '--protocol-root', PROTOCOL,
    '--expected-index-sha256', EXPECTED_INDEX_SHA256,
    '--expected-vector-index-sha256', EXPECTED_VECTOR_INDEX_SHA256,
]
subprocess.run([*base_command, '--output-root', SMOKE_OUTPUT, '--smoke'], check=True)
smoke_manifest = json.load(open(os.path.join(SMOKE_OUTPUT, 'probe_manifest.json'), encoding='utf-8'))
assert smoke_manifest['status'] == 'pass' and smoke_manifest['run_mode'] == 'smoke'
assert smoke_manifest['factors'] == ['length_words_whitespace_v1']
assert smoke_manifest['checks']['held_out_test_accessed'] is False
print({'real_data_probe_smoke': 'PASS', 'factors': smoke_manifest['factors'], 'bootstrap_replicates': smoke_manifest['bootstrap_replicates']})

In [ ]:
# Full development probe matrix. This is CPU-heavy but does not load raw EEG or train GLIM.
subprocess.run([*base_command, '--output-root', OUTPUT], check=True)

In [ ]:
import csv
manifest_path = os.path.join(OUTPUT, 'probe_manifest.json')
manifest = json.load(open(manifest_path, encoding='utf-8'))
assert manifest['status'] == 'pass' and manifest['run_mode'] == 'full_development'
assert manifest['source_index_sha256'] == EXPECTED_INDEX_SHA256
assert manifest['vector_index_sha256'] == EXPECTED_VECTOR_INDEX_SHA256
assert set(manifest['factors']) == {'sr_sentiment_3', 'nr_relation_content', 'tsr_instruction_relation', 'length_words_whitespace_v1'}
expected_checks = {
    'held_out_test_accessed': False, 'validation_tuning_permitted': False,
    'target_label_as_probe_input_permitted': False,
    'matched_wrong_model_fit_on_correct_training_vectors': True,
    'zero_and_gaussian_model_fit_on_correct_training_vectors': True,
    'subject_folds_refit_without_held_out_subject': True, 'null_results_retained': True,
}
assert manifest['checks'] == expected_checks
for name, expected in manifest['artifact_sha256'].items():
    assert digest(os.path.join(OUTPUT, name)) == expected, name
with open(os.path.join(OUTPUT, 'factor_admission.csv'), encoding='utf-8', newline='') as handle:
    admissions = list(csv.DictReader(handle))
assert len(admissions) == 4 and all(row['decision'] in {'admit', 'reject_retain_null'} for row in admissions)
shutil.copytree(PROTOCOL, os.path.join(OUTPUT, 'frozen_protocol'), dirs_exist_ok=True)
run_metadata = {
    'status': 'pass', 'project_commit': actual_commit, 'python': platform.python_version(),
    'numpy': numpy.__version__, 'scipy': scipy.__version__, 'scikit_learn': sklearn.__version__,
    'dataset_index_sha256': EXPECTED_INDEX_SHA256,
    'vector_index_sha256': EXPECTED_VECTOR_INDEX_SHA256, 'test_accessed': False,
}
with open(os.path.join(OUTPUT, 'run_metadata.json'), 'w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2, sort_keys=True)
    handle.write('\n')
print({'ordinary_point_gain_candidates': manifest['ordinary_point_gain_candidates'], 'admitted_factors': manifest['admitted_factors'], 'decisions': admissions})
for path in (WORKTREE, PROTOCOL, SMOKE_OUTPUT):
    if os.path.exists(path):
        shutil.rmtree(path)
print('FROZEN FACTOR RECOVERABILITY PROBES: PASS')